In [161]:
#!python -m pip install --upgrade pip
#%pip install pandas matplotlib seaborn scikit-learn openpyxl tensorflow xgboost aif360
#%pip install "aif360[Reductions, inFairness]"

In [162]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint
from collections import Counter
from scipy.stats import chi2_contingency, fisher_exact

from fairlearn.metrics import MetricFrame
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference, selection_rate, false_positive_rate, false_negative_rate, count
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

random_seed = 15

In [163]:
PATH = 'C:/Users/andre/Desktop/ProjectWork_AEQUITAS_AKKODIS/'

with open(PATH + 'data/predictions.json', 'r') as f:
    data = json.load(f)
predictions_df = pd.DataFrame(data['predictions'])
print(predictions_df.shape)
y_test = pd.Series(data['reference'])
print(y_test.shape)
s_test = pd.Series(data['sensitive'])
print(s_test.shape)
sensitive_feature = data['sensitive_name']
predictions_df.head(50)

(517, 15)
(517,)
(517,)


,Logistic Regression_preprocessed_cr,Linear Regression_preprocessed_cr,Decision Tree_preprocessed_cr,Naive Bayes_preprocessed_cr,XGBoost_preprocessed_cr,KNN_preprocessed_cr,Neural Network_preprocessed_cr,Linear Regression_inprocessed_gfc,Logistic Regression_postprocessed_to,Linear Regression_postprocessed_to,Decision Tree_postprocessed_to,Naive Bayes_postprocessed_to,XGBoost_postprocessed_to,KNN_postprocessed_to,Neural Network_postprocessed_to
0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
6,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
7,1,1,1,1,1,1,1,0,1,1,1,0,1,0,1
8,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
9,1,1,1,1,1,0,1,1,0,1,1,0,1,1,0


In [164]:
with open(PATH + 'data/encoding_mappings.json', 'r') as f:
    encoding_mappings = json.load(f)
pprint(encoding_mappings)

{'Age Range': {'20 - 25 years': 1,
               '26 - 30 years': 2,
               '31 - 35 years': 3,
               '36 - 40 years': 4,
               '40 - 45 years': 5,
               '< 20 years': 0,
               '> 45 years': 6},
 'Candidate State': {'Economic proposal': 5,
                     'First contact': 1,
                     'Hired': 6,
                     'Imported': 0,
                     'In selection': 2,
                     'QM': 3,
                     'Vivier': 4},
 'Current Ral': {'+ 50 K': 18,
                 '- 20 K': 2,
                 '20-22 K': 3,
                 '22-24 K': 4,
                 '24-26 K': 5,
                 '26-28 K': 6,
                 '28-30 K': 7,
                 '30-32 K': 8,
                 '32-34 K': 9,
                 '34-36 K': 10,
                 '36-38 K': 11,
                 '38-40 K': 12,
                 '40-42 K': 13,
                 '42-44 K': 14,
                 '44-46 K': 15,
                 '46-48 K': 16

## Fairness Metrics

In [165]:
metrics = []
for name in predictions_df.columns:
    y_pred = predictions_df[name]
    accuracy = round(accuracy_score(y_test, y_pred), 3)
    precision = round(precision_score(y_test, y_pred), 3)
    recall = round(recall_score(y_test, y_pred), 3)
    f1 = round(f1_score(y_test, y_pred), 3)
    roc_auc = round(roc_auc_score(y_test, y_pred), 3)

    metrics.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'ROC AUC': roc_auc
    })
metrics = pd.DataFrame(metrics)

In [166]:
def compute_fairness_metrics(y_true, y_pred, s_test, label=None):
    mf = MetricFrame(
        metrics={
            'selection_rate': selection_rate,
            'fpr': false_positive_rate,
            'fnr': false_negative_rate,
            'count': count
        },
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=s_test
    )

    dp = demographic_parity_difference(y_true, y_pred, sensitive_features=s_test)
    eo = equalized_odds_difference(y_true, y_pred, sensitive_features=s_test)

    if label:
        print(f"=== {label} ===")

    print("By group:")
    print(mf.by_group)
    print()
    print("Overall (selection_rate, fpr, fnr, count):")
    print(mf.overall)
    print()
    print(f"Demographic parity difference: {dp:.4f}")
    print(f"Equalized odds difference:     {eo:.4f}\n")

    return mf
for name in predictions_df.columns:
    compute_fairness_metrics(y_test, predictions_df[name], s_test, label=name)

=== Logistic Regression_preprocessed_cr ===
By group:
                     selection_rate       fpr       fnr  count
sensitive_feature_0                                           
0                          0.318584  0.000000  0.280000  113.0
1                          0.346535  0.015152  0.028571  404.0

Overall (selection_rate, fpr, fnr, count):
selection_rate      0.340426
fpr                 0.012232
fnr                 0.094737
count             517.000000
dtype: float64

Demographic parity difference: 0.0280
Equalized odds difference:     0.2514

=== Linear Regression_preprocessed_cr ===
By group:
                     selection_rate  fpr  fnr  count
sensitive_feature_0                                 
0                          0.442478  0.0  0.0  113.0
1                          0.346535  0.0  0.0  404.0

Overall (selection_rate, fpr, fnr, count):
selection_rate      0.367505
fpr                 0.000000
fnr                 0.000000
count             517.000000
dtype: float64

D

#### **3.1 Demographic Parity**

In [167]:
tolerance = 0.15
significance_level = 0.1

In [168]:
def calculate_demographic_parity(predictions, sensitive_attribute, name, significance_level, tolerance, activate_check=False):

    df = pd.DataFrame({
        'predictions': predictions,
        'sensitive_attribute': sensitive_attribute
    })
    prop = df.groupby('sensitive_attribute')['predictions'].mean()
    
    if activate_check:
        print(f"=== {name} ===")
        print(f"{prop}")

    if prop.shape[0] == 2:
        diff = prop.max() - prop.min()
        if activate_check:
            print(f"Two groups: |Δ| = {diff:.4f}, tol = {tolerance}")
        return 'T' if diff <= tolerance else False
    
    contingency_table = pd.crosstab(df['predictions'], df['sensitive_attribute'])
    chi2, p, dof, expected = chi2_contingency(contingency_table, correction=False)
    
    if contingency_table.shape == (2, 2) and (expected < 5).any():
        _, p = fisher_exact(contingency_table)
        if activate_check:
            print(f"Fisher’s exact test fallback for {name}")
    elif contingency_table.shape != (2, 2) and (expected < 5).any():
        if activate_check:
            print(f"Sparse contingency for {name}")
        
    return 'T' if p > significance_level else False    

table = []
for name in predictions_df.columns:
    result = calculate_demographic_parity(predictions_df[name], s_test, name, significance_level, tolerance, activate_check=True)
    table.append(result)
sf_df = pd.DataFrame(table, index = predictions_df.columns, columns=[sensitive_feature])

=== Logistic Regression_preprocessed_cr ===
sensitive_attribute
0    0.318584
1    0.346535
Name: predictions, dtype: float64
Two groups: |Δ| = 0.0280, tol = 0.15
=== Linear Regression_preprocessed_cr ===
sensitive_attribute
0    0.442478
1    0.346535
Name: predictions, dtype: float64
Two groups: |Δ| = 0.0959, tol = 0.15
=== Decision Tree_preprocessed_cr ===
sensitive_attribute
0    0.442478
1    0.346535
Name: predictions, dtype: float64
Two groups: |Δ| = 0.0959, tol = 0.15
=== Naive Bayes_preprocessed_cr ===
sensitive_attribute
0    0.442478
1    0.346535
Name: predictions, dtype: float64
Two groups: |Δ| = 0.0959, tol = 0.15
=== XGBoost_preprocessed_cr ===
sensitive_attribute
0    0.442478
1    0.346535
Name: predictions, dtype: float64
Two groups: |Δ| = 0.0959, tol = 0.15
=== KNN_preprocessed_cr ===
sensitive_attribute
0    0.256637
1    0.190594
Name: predictions, dtype: float64
Two groups: |Δ| = 0.0660, tol = 0.15
=== Neural Network_preprocessed_cr ===
sensitive_attribute
0    0.

#### **3.2 Equalized odds**

In [169]:
def calculate_equalized_odds(predictions, true_labels, sensitive_attribute, name, tolerance, activate_check=False):
    df = pd.DataFrame({
        'predictions': predictions,
        'true_labels': true_labels,
        'sensitive_attribute': sensitive_attribute
    })
    tprs, fprs = [], []
    for _, group_df in df.groupby('sensitive_attribute'):
        tn, fp, fn, tp = confusion_matrix(group_df['true_labels'], group_df['predictions'], labels=[0, 1]).ravel()
        tprs.append(tp / (tp + fn) if tp + fn != 0 else 0)
        fprs.append(fp / (fp + tn) if fp + tn != 0 else 0)

    max_tpr_diff = max(tprs) - min(tprs)
    max_fpr_diff = max(fprs) - min(fprs)

    if activate_check:
            print(f"=== {name} ===")
            print(f"Max FPR difference: {max_fpr_diff}")
            print(f"Max TPR difference: {max_tpr_diff}")

    return 'T' if (max_tpr_diff <= 2 * tolerance and max_fpr_diff <= 2 * tolerance) else False


table = []
for name in predictions_df.columns:
    result = calculate_equalized_odds(predictions_df[name], y_test, s_test, name, tolerance, activate_check=True)
    table.append(result)
sf_df = pd.DataFrame(table, index = predictions_df.columns, columns=[sensitive_feature])

=== Logistic Regression_preprocessed_cr ===
Max FPR difference: 0.015151515151515152
Max TPR difference: 0.25142857142857145
=== Linear Regression_preprocessed_cr ===
Max FPR difference: 0.0
Max TPR difference: 0.0
=== Decision Tree_preprocessed_cr ===
Max FPR difference: 0.0
Max TPR difference: 0.0
=== Naive Bayes_preprocessed_cr ===
Max FPR difference: 0.0
Max TPR difference: 0.0
=== XGBoost_preprocessed_cr ===
Max FPR difference: 0.0
Max TPR difference: 0.0
=== KNN_preprocessed_cr ===
Max FPR difference: 0.01713564213564213
Max TPR difference: 0.07714285714285712
=== Neural Network_preprocessed_cr ===
Max FPR difference: 0.01984126984126977
Max TPR difference: 0.020000000000000018
=== Linear Regression_inprocessed_gfc ===
Max FPR difference: 0.003787878787878788
Max TPR difference: 0.07714285714285718
=== Logistic Regression_postprocessed_to ===
Max FPR difference: 0.003787878787878788
Max TPR difference: 0.03142857142857147
=== Linear Regression_postprocessed_to ===
Max FPR differe